# ROGII Wellbore Geology: Baseline + Exploration Walkthrough

This notebook is a public, readable baseline for the ROGII Wellbore Geology Prediction competition. It is not meant to be mysterious or leaderboard-maxed. The goal is to show a practical path from first data inspection to a leakage-aware LightGBM baseline that produces a valid `submission.csv`.

The central idea: treat each well as an ordered sequence, keep validation grouped by well, create simple geometry and GR history features, add typewell context, then smooth predictions before writing the submission.


## Provenance and attribution

This notebook was written from scratch for this ROGII project. It uses standard pandas, scikit-learn, and LightGBM patterns, but it does not copy or adapt code from a specific public Kaggle notebook. If I later incorporate a concrete idea or code pattern from another public notebook, I will cite it in this section.


## What each project file is for

- `kaggle/public_baseline_exploration/rogii_baseline_exploration.ipynb`: this public walkthrough notebook. It explains the baseline and runs fully on Kaggle.
- `kaggle/public_baseline_exploration/kernel-metadata.json`: Kaggle metadata for this public notebook, including its displayed title and competition data source.
- `kaggle/rogii_remote/rogii_remote.py`: the private remote-run script used for the first submitted baseline. It is kept script-style for reliable automation.
- `kaggle/rogii_remote/kernel-metadata.json`: Kaggle metadata for the private remote baseline kernel.
- `scripts/kaggle_remote.py`: local orchestration only. It downloads/list/push/poll/collect/submit; it does not train models locally.
- `README.md`: project operating notes, especially the rule that heavy compute belongs on Kaggle.

The public notebook repeats the important modeling logic from the private baseline, but with more narrative notes so other competitors can learn from the sequence of choices.


## Baseline path

This is the progression used here:

1. Inspect the input layout and sample submission.
2. Build row-level features from measured depth, XYZ position, GR, and known `TVT_input`.
3. Add typewell summaries and an interpolated typewell GR signal.
4. Validate with `GroupKFold` by `well_id`, so rows from the same well do not appear in both train and validation.
5. Train a plain LightGBM regressor.
6. Smooth the test predictions within each well and write `submission.csv`.

This is intentionally a baseline. Good next improvements are listed at the end.


In [ ]:
from __future__ import annotations

import gc
import json
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold

warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
    HAS_LIGHTGBM = True
except Exception:
    from sklearn.ensemble import HistGradientBoostingRegressor
    HAS_LIGHTGBM = False

COMPETITION = "rogii-wellbore-geology-prediction"
INPUT_BASE = Path("/kaggle/input")
WORKING = Path("/kaggle/working")
TARGET = "TVT"
RANDOM_STATE = 42
N_FOLDS = 5
EXTRA_GR_GAPS = (2, 10, 50)

HORIZONTAL_COLS = ["MD", "X", "Y", "Z", "GR", "TVT_input", "TVT"]
TYPEWELL_COLS = ["TVT", "GR"]

print(f"LightGBM available: {HAS_LIGHTGBM}")


## 1. Find the competition input

Kaggle can mount competition data in slightly different paths depending on API version and notebook environment. This helper finds the directory containing `sample_submission.csv`, `train/`, and `test/`. If Kaggle ever mounts only a zip, the fallback extracts it inside `/kaggle/working`.


In [ ]:
def shallow_tree(root: Path, max_items: int = 80) -> list[str]:
    if not root.exists():
        return [f"{root} does not exist"]
    lines = []
    for i, path in enumerate(sorted(root.rglob("*"))):
        if i >= max_items:
            lines.append(f"... truncated after {max_items} items")
            break
        rel = path.relative_to(root) if path.is_relative_to(root) else path
        suffix = "/" if path.is_dir() else ""
        lines.append(f"{rel}{suffix}")
    return lines


def find_input_root() -> Path:
    candidates = [
        INPUT_BASE / COMPETITION,
        INPUT_BASE / "competitions" / COMPETITION,
        INPUT_BASE / "ROGII - Wellbore Geology Prediction",
    ]
    candidates.extend(path for path in sorted(INPUT_BASE.glob("*")) if path.is_dir())
    candidates.extend(path.parent for path in sorted(INPUT_BASE.rglob("sample_submission.csv")))

    seen = set()
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "sample_submission.csv").exists() and (candidate / "train").is_dir():
            return candidate

    zip_candidates = sorted(INPUT_BASE.rglob(f"{COMPETITION}.zip"))
    zip_candidates.extend(sorted(INPUT_BASE.rglob("*.zip")))
    for zip_path in zip_candidates:
        target = WORKING / "competition_input"
        target.mkdir(parents=True, exist_ok=True)
        marker = target / ".extracted"
        if not marker.exists():
            print(f"Extracting mounted archive: {zip_path}")
            with zipfile.ZipFile(zip_path) as zf:
                zf.extractall(target)
            marker.write_text(str(zip_path), encoding="utf-8")
        if (target / "sample_submission.csv").exists() and (target / "train").is_dir():
            return target

    print("Could not locate competition input. /kaggle/input contains:")
    for line in shallow_tree(INPUT_BASE):
        print(f"  {line}")
    raise FileNotFoundError("Competition input root not found")


INPUT = find_input_root()
TRAIN_DIR = INPUT / "train"
TEST_DIR = INPUT / "test"
print(INPUT)

## 2. Quick data inspection

The sample submission tells us the required IDs and target column. Train horizontal wells contain `TVT`, while test horizontal wells contain known `TVT_input` before the prediction-start region and blanks afterward.


In [ ]:
sample = pd.read_csv(INPUT / "sample_submission.csv")
train_paths = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
test_paths = sorted(TEST_DIR.glob("*__horizontal_well.csv"))

print(f"train horizontal wells: {len(train_paths)}")
print(f"test horizontal wells: {len(test_paths)}")
display(sample.head())
display(pd.read_csv(train_paths[0]).head())
display(pd.read_csv(test_paths[0]).head())

## 3. Feature engineering

This is the main baseline feature block. For each well, we create:

- row identity: `well_id`, `row_idx`, and submission `id`
- known-TVT indicators, forward-filled `TVT_input`, and row distances to known TVT anchors
- measured-depth and XYZ offsets from the start and from the prediction-start anchor
- horizontal and 3D distance from the prediction-start anchor
- simple trajectory derivatives such as `dz_dmd`
- GR lag, gap-diff, and rolling statistics
- typewell summary features and an interpolated typewell GR value at the current TVT context

These are deliberately transparent features. They are easy to inspect and form a good public baseline before moving to more specialized geology alignment, target-parameterization, or ensemble methods.


In [ ]:
def rmse(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def well_id_from_path(path: Path) -> str:
    return path.name.split("__", 1)[0]


def safe_read_csv(path: Path, keep_cols: list[str]) -> pd.DataFrame:
    return pd.read_csv(path, usecols=lambda col: col in keep_cols)


def typewell_frame(path: Path) -> pd.DataFrame:
    tw = safe_read_csv(path, TYPEWELL_COLS)
    for col in TYPEWELL_COLS:
        if col not in tw.columns:
            tw[col] = np.nan
    return tw.dropna(subset=["TVT"]).sort_values("TVT").reset_index(drop=True)


def add_typewell_features(df: pd.DataFrame, tw: pd.DataFrame) -> pd.DataFrame:
    tw_gr = pd.to_numeric(tw["GR"], errors="coerce")
    tw_tvt = pd.to_numeric(tw["TVT"], errors="coerce")

    stats = {
        "tw_tvt_min": tw_tvt.min(),
        "tw_tvt_max": tw_tvt.max(),
        "tw_tvt_span": tw_tvt.max() - tw_tvt.min(),
        "tw_gr_mean": tw_gr.mean(),
        "tw_gr_std": tw_gr.std(),
        "tw_gr_min": tw_gr.min(),
        "tw_gr_max": tw_gr.max(),
        "tw_gr_q10": tw_gr.quantile(0.10),
        "tw_gr_q50": tw_gr.quantile(0.50),
        "tw_gr_q90": tw_gr.quantile(0.90),
    }
    for name, value in stats.items():
        df[name] = value

    valid = tw[["TVT", "GR"]].dropna()
    if len(valid) >= 2:
        df["tw_gr_at_tvt_input_ffill"] = np.interp(
            df["tvt_input_ffill"].to_numpy(dtype="float64"),
            valid["TVT"].to_numpy(dtype="float64"),
            valid["GR"].to_numpy(dtype="float64"),
            left=np.nan,
            right=np.nan,
        )
    else:
        df["tw_gr_at_tvt_input_ffill"] = np.nan
    return df


def engineer_one_well(path: Path) -> pd.DataFrame:
    well_id = well_id_from_path(path)
    df = safe_read_csv(path, HORIZONTAL_COLS)
    for col in HORIZONTAL_COLS:
        if col not in df.columns:
            df[col] = np.nan

    df["well_id"] = well_id
    df["row_idx"] = np.arange(len(df), dtype=np.int32)
    df["id"] = well_id + "_" + df["row_idx"].astype(str)

    for col in ["MD", "X", "Y", "Z", "GR", "TVT_input", "TVT"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    known = df["TVT_input"].notna()
    last_known_idx = int(np.flatnonzero(known.to_numpy()).max()) if known.any() else 0
    first = df.iloc[0]
    anchor = df.iloc[last_known_idx]

    df["tvt_input_known"] = known.astype("float32")
    df["tvt_input_ffill"] = df["TVT_input"].ffill()
    df["tvt_input_bfill"] = df["TVT_input"].bfill()
    df["tvt_input_delta"] = df["tvt_input_ffill"].diff()

    known_idx = np.where(known.to_numpy(), df["row_idx"].to_numpy(dtype="float64"), np.nan)
    prev_known = pd.Series(known_idx).ffill()
    next_known = pd.Series(known_idx).bfill()
    df["rows_since_tvt_input"] = df["row_idx"] - prev_known
    df["rows_until_tvt_input"] = next_known - df["row_idx"]

    df["md_from_start"] = df["MD"] - first["MD"]
    df["x_from_start"] = df["X"] - first["X"]
    df["y_from_start"] = df["Y"] - first["Y"]
    df["z_from_start"] = df["Z"] - first["Z"]
    df["md_from_ps"] = df["MD"] - anchor["MD"]
    df["idx_from_ps"] = df["row_idx"] - last_known_idx
    df["x_from_ps"] = df["X"] - anchor["X"]
    df["y_from_ps"] = df["Y"] - anchor["Y"]
    df["z_from_ps"] = df["Z"] - anchor["Z"]
    df["xy_dist_from_ps"] = np.sqrt(df["x_from_ps"] * df["x_from_ps"] + df["y_from_ps"] * df["y_from_ps"])
    df["xyz_dist_from_ps"] = np.sqrt(
        df["x_from_ps"] * df["x_from_ps"] + df["y_from_ps"] * df["y_from_ps"] + df["z_from_ps"] * df["z_from_ps"]
    )

    dx = df["X"].diff()
    dy = df["Y"].diff()
    dz = df["Z"].diff()
    dmd = df["MD"].diff()
    df["step_3d"] = np.sqrt(dx * dx + dy * dy + dz * dz)
    df["cum_3d"] = df["step_3d"].fillna(0).cumsum()
    df["dz_dmd"] = dz / dmd.replace(0, np.nan)
    df["dx_dmd"] = dx / dmd.replace(0, np.nan)
    df["dy_dmd"] = dy / dmd.replace(0, np.nan)

    gr = df["GR"]
    df["gr_diff1"] = gr.diff()
    df["gr_lag1"] = gr.shift(1)
    df["gr_lag5"] = gr.shift(5)
    df["gr_lag25"] = gr.shift(25)
    for gap in EXTRA_GR_GAPS:
        df[f"gr_diff{gap}"] = gr.diff(gap)
        df[f"gr_absdiff{gap}"] = df[f"gr_diff{gap}"].abs()
    for window in (5, 25, 101):
        roll = gr.rolling(window, min_periods=1)
        df[f"gr_roll_mean_{window}"] = roll.mean()
        df[f"gr_roll_std_{window}"] = roll.std()
        df[f"gr_roll_min_{window}"] = roll.min()
        df[f"gr_roll_max_{window}"] = roll.max()

    typewell_path = path.parent / f"{well_id}__typewell.csv"
    tw = typewell_frame(typewell_path) if typewell_path.exists() else pd.DataFrame({"TVT": [], "GR": []})
    return add_typewell_features(df, tw)


## 4. Load train and test wells

This is the first heavier step. On Kaggle, it loads all horizontal wells and builds the feature matrix. Keeping this work inside the notebook is intentional: the project rule is that heavy computation happens on Kaggle, not locally.


In [ ]:
def load_split(split: str) -> pd.DataFrame:
    base_dir = TRAIN_DIR if split == "train" else TEST_DIR
    paths = sorted(base_dir.glob("*__horizontal_well.csv"))
    print(f"Loading {split}: {len(paths)} horizontal wells")
    if not paths:
        raise FileNotFoundError(f"No {split} horizontal well files found under {base_dir}")

    frames = []
    for i, path in enumerate(paths, start=1):
        if i % 50 == 0 or i == len(paths):
            print(f"  {split}: {i}/{len(paths)}")
        frames.append(engineer_one_well(path))
    out = pd.concat(frames, ignore_index=True)
    print(f"{split} shape: {out.shape}")
    return out


def feature_columns(train: pd.DataFrame, test: pd.DataFrame) -> list[str]:
    excluded = {"TVT", "well_id", "id"}
    common = [col for col in train.columns if col in test.columns and col not in excluded]
    return [col for col in common if pd.api.types.is_numeric_dtype(train[col])]


train = load_split("train")
test = load_split("test")
features = feature_columns(train, test)
print(f"Using {len(features)} features")
features

## 5. Grouped validation and LightGBM baseline

Rows inside the same well are strongly related, so random row-level validation would be too optimistic. `GroupKFold` keeps each `well_id` entirely inside either train or validation for a fold. This gives a better read on whether the model generalizes to held-out wells.


In [ ]:
def as_matrix(df: pd.DataFrame, feature_names: list[str]) -> pd.DataFrame:
    return df[feature_names].astype("float32")


def make_model() -> object:
    if HAS_LIGHTGBM:
        return lgb.LGBMRegressor(
            objective="regression",
            metric="rmse",
            n_estimators=2500,
            learning_rate=0.035,
            num_leaves=96,
            subsample=0.85,
            subsample_freq=1,
            colsample_bytree=0.85,
            reg_lambda=5.0,
            min_child_samples=80,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=-1,
        )
    return HistGradientBoostingRegressor(
        loss="squared_error",
        learning_rate=0.06,
        max_iter=350,
        l2_regularization=1.0,
        random_state=RANDOM_STATE,
    )


def fit_predict(train: pd.DataFrame, test: pd.DataFrame, feature_names: list[str]):
    target_mask = train[TARGET].notna().to_numpy()
    fit_df = train.loc[target_mask].reset_index(drop=True)
    groups = fit_df["well_id"].to_numpy()
    y = fit_df[TARGET].to_numpy(dtype="float32")
    x_test = as_matrix(test, feature_names)

    oof = np.zeros(len(fit_df), dtype="float32")
    test_pred = np.zeros(len(test), dtype="float32")
    models = []

    splitter = GroupKFold(n_splits=N_FOLDS)
    for fold, (tr_idx, va_idx) in enumerate(splitter.split(fit_df, y, groups), start=1):
        print(f"Fold {fold}/{N_FOLDS}: train={len(tr_idx)} valid={len(va_idx)}")
        x_tr = as_matrix(fit_df.iloc[tr_idx], feature_names)
        y_tr = y[tr_idx]
        x_va = as_matrix(fit_df.iloc[va_idx], feature_names)
        y_va = y[va_idx]

        model = make_model()
        if HAS_LIGHTGBM:
            model.fit(
                x_tr,
                y_tr,
                eval_set=[(x_va, y_va)],
                eval_metric="rmse",
                callbacks=[lgb.early_stopping(120), lgb.log_evaluation(100)],
            )
        else:
            model.fit(x_tr, y_tr)

        va_pred = model.predict(x_va).astype("float32")
        fold_test_pred = model.predict(x_test).astype("float32")
        oof[va_idx] = va_pred
        test_pred += fold_test_pred / N_FOLDS
        print(f"Fold {fold} RMSE: {rmse(y_va, va_pred):.5f}")

        models.append(model)
        del x_tr, x_va, y_tr, y_va, va_pred, fold_test_pred
        gc.collect()

    fit_df["_oof_pred"] = oof
    fit_df["_target_used"] = y
    return fit_df, oof, test_pred, models


fit_df, oof, raw_test_pred, models = fit_predict(train, test, features)

## 6. Metrics, smoothing, and submission

The model predicts row by row, but geology should not jump wildly from one measured-depth row to the next. A small exponential smoothing pass by well is a simple postprocess that makes the test curve more physically plausible. Known `TVT_input` rows are kept anchored.


In [ ]:
def smooth_test_predictions(test: pd.DataFrame, pred: np.ndarray) -> pd.Series:
    work = test[["well_id", "row_idx", "TVT_input", "tw_tvt_min", "tw_tvt_max"]].copy()
    work["pred"] = pred
    known = work["TVT_input"].notna()
    work.loc[known, "pred"] = work.loc[known, "TVT_input"]

    pieces = []
    for _, group in work.sort_values(["well_id", "row_idx"]).groupby("well_id", sort=False):
        smoothed = group["pred"].ewm(alpha=0.35, adjust=False).mean()
        lo = group["tw_tvt_min"].iloc[0]
        hi = group["tw_tvt_max"].iloc[0]
        if np.isfinite(lo) and np.isfinite(hi):
            smoothed = smoothed.clip(lo - 500.0, hi + 500.0)
        pieces.append(pd.Series(smoothed.to_numpy(), index=group.index))
    return pd.concat(pieces).sort_index()


def write_feature_importance(models: list[object], feature_names: list[str]) -> None:
    if not HAS_LIGHTGBM:
        pd.DataFrame({"feature": feature_names}).to_csv("feature_importance.csv", index=False)
        return

    rows = []
    for fold, model in enumerate(models, start=1):
        booster = model.booster_
        for kind in ("gain", "split"):
            values = booster.feature_importance(importance_type=kind)
            rows.extend(
                {"fold": fold, "feature": feature, "importance_type": kind, "importance": value}
                for feature, value in zip(feature_names, values)
            )
    pd.DataFrame(rows).to_csv("feature_importance.csv", index=False)


used = train[TARGET].notna().to_numpy()
y_used = train.loc[used, TARGET].to_numpy(dtype="float32")
tvt_input_missing = train.loc[used, "TVT_input"].isna().to_numpy()

metrics = {
    "n_train_rows": int(used.sum()),
    "n_test_rows": int(len(test)),
    "n_features": int(len(features)),
    "n_folds": int(N_FOLDS),
    "model": "lightgbm" if HAS_LIGHTGBM else "hist_gradient_boosting",
    "oof_rmse_all": rmse(y_used, oof),
    "oof_rmse_prediction_region": rmse(y_used[tvt_input_missing], oof[tvt_input_missing])
    if tvt_input_missing.any()
    else None,
}

print(json.dumps(metrics, indent=2))

sample = pd.read_csv(INPUT / "sample_submission.csv")
test["pred_raw"] = raw_test_pred
test["pred"] = smooth_test_predictions(test, raw_test_pred).astype("float32")

pred_map = test.set_index("id")["pred"]
fallback = float(np.nanmean(raw_test_pred)) if len(raw_test_pred) else float(np.nanmean(y_used))
submission = sample.copy()
submission["tvt"] = submission["id"].map(pred_map).fillna(fallback).astype("float32")
submission.to_csv("submission.csv", index=False)

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
write_feature_importance(models, features)

display(submission.head())
print("Wrote submission.csv, metrics.json, feature_importance.csv")

## 7. What improved the baseline along the way

The practical improvements in this public baseline are small but cumulative:

- **Leakage control:** grouped validation by `well_id` avoids row-level leakage.
- **Prediction-start anchoring:** offsets, distances, and row counts from known `TVT_input` rows help the model understand where extrapolation begins.
- **Trajectory features:** XYZ and measured-depth derivatives capture the geometry of the horizontal well.
- **GR history:** lag, gap-diff, and rolling GR features add local sequence context without a neural sequence model.
- **Typewell context:** typewell GR and TVT summaries give the model nearby geology hints.
- **Smoothing:** a light per-well postprocess removes noisy row-to-row jumps.

This public version intentionally stays with a simple absolute-TVT LightGBM baseline. Later private experiments can test target deltas, stronger curve alignment, XGBoost/CatBoost variants, and OOF-based blends without mixing those discoveries into this shareable notebook.

Next directions I would explore:

1. Stronger alignment between horizontal GR curves and typewell GR curves.
2. Separate models for known-input region versus prediction region.
3. More careful validation splits that mimic the hidden test wells.
4. LightGBM/CatBoost ensembles with OOF-based blending.
5. Monotonic or physics-aware postprocessing so TVT curves stay geologically plausible.
